# CardioSafe AI - Notebook 02: baselines (classical ML + 1D CNN)

Trains four classical models on handcrafted features and a plain 1D CNN (3 seeds) on the raw signal, all with the same evaluation, so later models (1D ResNet etc.) are compared fairly.

**Runtime: set to GPU (T4).** Needs `ptbxl_100hz_super5.npz` from notebook 01 in `MyDrive/CardioSafe`.

Evaluation (same for every model): macro AUROC with bootstrap 95% CI on the test fold (10), macro AUPRC, macro F1 with per-class thresholds tuned on the validation fold (9), plus per-class numbers. Predictions are saved to Drive so calibration, ensembling and explainability notebooks can reuse them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
WORK = '/content/drive/MyDrive/CardioSafe'
PRED = WORK + '/preds'
os.makedirs(PRED, exist_ok=True)
sys.path.insert(0, WORK)
print(os.listdir(WORK))

### Shared modules (written to Drive so every later notebook can import them)

In [ ]:
%%writefile /content/drive/MyDrive/CardioSafe/cardiosafe_common.py
"""Shared utilities for CardioSafe AI notebooks: data loading, evaluation, deep-model training."""
import os, copy, time
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

CLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']


# ----------------------------------------------------------------- data
def load_data(path):
    z = np.load(path)
    X, Y, fold = z['X'], z['Y'], z['fold']
    tr, va, te = fold <= 8, fold == 9, fold == 10
    return X, Y, fold, tr, va, te, z


# ----------------------------------------------------------- evaluation
def tune_thresholds(y, p, grid=None):
    """Per-class threshold maximising F1 on the validation set."""
    grid = np.linspace(0.02, 0.98, 97) if grid is None else grid
    ths = []
    for c in range(y.shape[1]):
        f = [f1_score(y[:, c], p[:, c] >= t, zero_division=0) for t in grid]
        ths.append(grid[int(np.argmax(f))])
    return np.array(ths)


def macro_auc(y, p):
    return float(np.mean([roc_auc_score(y[:, c], p[:, c]) for c in range(y.shape[1])]))


def bootstrap_ci(y, p, n=500, seed=0, alpha=0.05):
    rng = np.random.default_rng(seed)
    N, vals = len(y), []
    for _ in range(n):
        idx = rng.integers(0, N, N)
        try:
            vals.append(macro_auc(y[idx], p[idx]))
        except ValueError:      # a class missing in the resample
            continue
    return float(np.quantile(vals, alpha / 2)), float(np.quantile(vals, 1 - alpha / 2))


def summarize(name, y_val, p_val, y_te, p_te, save_dir=None, ci=True):
    """Metrics on test, with thresholds tuned on validation. Optionally saves predictions."""
    ths = tune_thresholds(y_val, p_val)
    pred = p_te >= ths
    per_auc = [roc_auc_score(y_te[:, c], p_te[:, c]) for c in range(y_te.shape[1])]
    per_ap = [average_precision_score(y_te[:, c], p_te[:, c]) for c in range(y_te.shape[1])]
    per_f1 = [f1_score(y_te[:, c], pred[:, c], zero_division=0) for c in range(y_te.shape[1])]
    row = {'model': name, 'AUROC': np.mean(per_auc), 'AUPRC': np.mean(per_ap), 'F1': np.mean(per_f1)}
    if ci:
        lo, hi = bootstrap_ci(y_te, p_te)
        row['AUROC_lo'], row['AUROC_hi'] = lo, hi
    for c, n in enumerate(CLASSES):
        row[f'AUROC_{n}'] = per_auc[c]
    for c, n in enumerate(CLASSES):
        row[f'F1_{n}'] = per_f1[c]
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        np.savez(os.path.join(save_dir, f'{name}.npz'), val=p_val, test=p_te, thresholds=ths)
    return row


# --------------------------------------------------------- deep learning
def standardize_fit(X_train):
    """Per-lead mean/std from the training set. X shape (N, T, 12)."""
    mu = X_train.mean(axis=(0, 1), keepdims=True)
    sd = X_train.std(axis=(0, 1), keepdims=True) + 1e-6
    return mu.astype(np.float32), sd.astype(np.float32)


def to_tensor_ncl(X, mu, sd, device):
    """(N, T, 12) numpy -> (N, 12, T) float tensor on device, standardised."""
    Xn = ((X - mu) / sd).transpose(0, 2, 1).astype(np.float32)
    return torch.from_numpy(Xn).to(device)


def set_seed(seed):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def predict(model, Xt, bs=512):
    model.eval()
    out = []
    for i in range(0, len(Xt), bs):
        out.append(torch.sigmoid(model(Xt[i:i + bs]).float()).cpu())
    return torch.cat(out).numpy()


def train_deep(model, Xtr, Ytr, Xva, Yva, epochs=40, lr=2e-3, wd=1e-2, bs=128,
               seed=0, augment=None, loss_fn=None, verbose=True):
    """Train with AdamW + OneCycle + AMP; keep the epoch with best validation macro AUROC.
    Xtr/Xva: tensors (N,12,T) on device. Ytr: float tensor (N,C) on device. Yva: numpy (N,C)."""
    set_seed(seed)
    device = Xtr.device
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    steps = int(np.ceil(len(Xtr) / bs)) * epochs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.2)
    loss_fn = loss_fn or nn.BCEWithLogitsLoss()
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    best, best_state, hist = -1, None, []
    for ep in range(epochs):
        model.train()
        t0 = time.time()
        perm = torch.randperm(len(Xtr), device=device)
        tot = 0.0
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]
            xb, yb = Xtr[idx], Ytr[idx]
            if augment is not None:
                xb = augment(xb)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = loss_fn(model(xb).float(), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(idx)
        pv = predict(model, Xva)
        auc = macro_auc(Yva, pv)
        hist.append((ep, tot / len(Xtr), auc))
        if auc > best:
            best, best_state = auc, copy.deepcopy(model.state_dict())
        if verbose:
            print(f'ep {ep + 1:02d}/{epochs}  loss {tot / len(Xtr):.4f}  val AUROC {auc:.4f}  best {best:.4f}  ({time.time() - t0:.0f}s)')
    model.load_state_dict(best_state)
    return model, hist


# ----------------------------------------------------------- baseline CNN
def make_cnn1d(n_classes=5, in_ch=12, width=32, p_drop=0.3):
    class GlobalPool(nn.Module):
        def forward(self, x):
            return torch.cat([x.mean(-1), x.amax(-1)], dim=1)

    def block(i, o, k):
        return nn.Sequential(nn.Conv1d(i, o, k, padding=k // 2, bias=False), nn.BatchNorm1d(o),
                             nn.ReLU(inplace=True), nn.MaxPool1d(2))

    w = width
    return nn.Sequential(
        block(in_ch, w, 15), block(w, 2 * w, 9), block(2 * w, 4 * w, 7), block(4 * w, 8 * w, 5),
        GlobalPool(), nn.Dropout(p_drop), nn.Linear(16 * w, n_classes))


In [ ]:
%%writefile /content/drive/MyDrive/CardioSafe/cardiosafe_features.py
"""Fast handcrafted ECG features for classical ML baselines. Input X: (N, T, 12) at 100 Hz."""
import numpy as np
from scipy.signal import butter, filtfilt, find_peaks
from scipy.stats import skew, kurtosis

FS = 100
BANDS = [(0.5, 3), (3, 10), (10, 20), (20, 40)]


def _stat_feats(x):
    """x: (n, T, 12) -> (n, 12*F) amplitude/shape/spectral features per lead."""
    n, T, L = x.shape
    f = [x.mean(1), x.std(1), x.min(1), x.max(1), np.ptp(x, axis=1),
         skew(x, axis=1), kurtosis(x, axis=1)]
    q = np.percentile(x, [5, 25, 50, 75, 95], axis=1)           # (5, n, 12)
    f += [q[i] for i in range(5)]
    d = np.diff(x, axis=1)
    f += [np.abs(d).mean(1), d.std(1)]                          # roughness / slope
    spec = np.abs(np.fft.rfft(x - x.mean(1, keepdims=True), axis=1)) ** 2
    freqs = np.fft.rfftfreq(T, 1 / FS)
    total = spec.sum(1) + 1e-9
    for lo, hi in BANDS:
        m = (freqs >= lo) & (freqs < hi)
        f.append(spec[:, m].sum(1) / total)
    return np.concatenate(f, axis=1)


_b, _a = butter(2, [5 / (FS / 2), 15 / (FS / 2)], btype='band')


def _hrv_feats(x):
    """Crude QRS detection on lead II (index 1) + RR-interval statistics."""
    out = np.zeros((len(x), 9), dtype=np.float32)
    for i in range(len(x)):
        s = filtfilt(_b, _a, x[i, :, 1]) ** 2
        thr = 0.35 * np.percentile(s, 99)
        pk, _ = find_peaks(s, height=thr, distance=int(0.25 * FS))
        if len(pk) < 3:
            out[i, 0] = len(pk)
            continue
        rr = np.diff(pk) / FS
        d = np.abs(np.diff(rr))
        out[i] = [len(pk), 60 / rr.mean(), rr.mean(), rr.std(), np.sqrt((np.diff(rr) ** 2).mean()),
                  (d > 0.05).mean(), rr.min(), rr.max(), rr.std() / rr.mean()]
    return out


HRV_NAMES = ['n_beats', 'hr_bpm', 'rr_mean', 'sdnn', 'rmssd', 'pnn50', 'rr_min', 'rr_max', 'rr_cv']
STAT_NAMES = (['mean', 'std', 'min', 'max', 'ptp', 'skew', 'kurt', 'p5', 'p25', 'p50', 'p75', 'p95',
               'absdiff', 'diffstd'] + [f'band{lo}-{hi}' for lo, hi in BANDS])


def feature_names(leads):
    return [f'{s}_{l}' for s in STAT_NAMES for l in leads] + HRV_NAMES


def extract_features(X, chunk=500, progress=None):
    parts = []
    it = range(0, len(X), chunk)
    if progress:
        it = progress(it)
    for i in it:
        xc = X[i:i + chunk].astype(np.float64)
        parts.append(np.concatenate([_stat_feats(xc), _hrv_feats(xc)], axis=1).astype(np.float32))
    F = np.concatenate(parts)
    return np.nan_to_num(F, nan=0.0, posinf=0.0, neginf=0.0)


In [ ]:
import importlib, numpy as np, pandas as pd, torch, time
import cardiosafe_common as C, cardiosafe_features as F
importlib.reload(C); importlib.reload(F)
from tqdm.auto import tqdm

X, Y, fold, tr, va, te, meta = C.load_data(WORK + '/ptbxl_100hz_super5.npz')
print('X', X.shape, '| train/val/test:', tr.sum(), va.sum(), te.sum())
print('device:', 'GPU ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (switch runtime to GPU for the CNN)')
results = []

## 1. Handcrafted features (statistics, spectral band power, QRS-based heart-rate variability)

In [ ]:
feat_path = WORK + '/features_super5.npy'
if os.path.exists(feat_path):
    Fm = np.load(feat_path)
else:
    t0 = time.time()
    Fm = F.extract_features(X, progress=tqdm)
    np.save(feat_path, Fm)
    print(f'{time.time() - t0:.0f}s')
names = F.feature_names(C.LEADS)
print('Feature matrix:', Fm.shape, '| names:', len(names))

## 2. Classical models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

def proba_list(m, A):          # multi-label sklearn models return one (N,2) array per class
    return np.stack([q[:, 1] for q in m.predict_proba(A)], axis=1)

def run(name, fit_predict):
    t0 = time.time()
    pv, pt = fit_predict()
    row = C.summarize(name, Y[va], pv, Y[te], pt, save_dir=PRED)
    row['train_s'] = round(time.time() - t0)
    results.append(row)
    print(f"{name:22s} AUROC {row['AUROC']:.4f} [{row['AUROC_lo']:.4f}, {row['AUROC_hi']:.4f}]  F1 {row['F1']:.4f}  ({row['train_s']}s)")

def lr_fp():
    m = OneVsRestClassifier(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.05, class_weight='balanced')), n_jobs=-1)
    m.fit(Fm[tr], Y[tr]); return m.predict_proba(Fm[va]), m.predict_proba(Fm[te])

def dt_fp():
    m = DecisionTreeClassifier(max_depth=8, min_samples_leaf=20, class_weight='balanced', random_state=0).fit(Fm[tr], Y[tr])
    return proba_list(m, Fm[va]), proba_list(m, Fm[te])

def rf_fp():
    m = RandomForestClassifier(n_estimators=500, min_samples_leaf=3, class_weight='balanced_subsample', n_jobs=-1, random_state=0).fit(Fm[tr], Y[tr])
    return proba_list(m, Fm[va]), proba_list(m, Fm[te])

def xgb_fp():
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    pv, pt = np.zeros((va.sum(), 5)), np.zeros((te.sum(), 5))
    for c in range(5):
        m = xgb.XGBClassifier(n_estimators=1500, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.6,
                              tree_method='hist', device=dev, early_stopping_rounds=60, eval_metric='logloss', random_state=0)
        m.fit(Fm[tr], Y[tr][:, c], eval_set=[(Fm[va], Y[va][:, c])], verbose=False)
        pv[:, c], pt[:, c] = m.predict_proba(Fm[va])[:, 1], m.predict_proba(Fm[te])[:, 1]
    return pv, pt

run('logreg', lr_fp)
run('decision_tree', dt_fp)
run('random_forest', rf_fp)
run('xgboost', xgb_fp)

## 3. Plain 1D CNN on the raw 12-lead signal
Three seeds, so we can report mean ± std instead of one lucky run. Roughly 5-8 minutes per seed on a T4.

In [ ]:
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mu, sd = C.standardize_fit(X[tr])
np.savez(WORK + '/norm_stats.npz', mu=mu, sd=sd)
Xtr, Xva, Xte = [C.to_tensor_ncl(X[m], mu, sd, dev) for m in (tr, va, te)]
Ytr = torch.tensor(Y[tr]).to(dev)

SEEDS = [0, 1, 2]
cnn_val, cnn_test = [], []
for s in SEEDS:
    print(f'===== 1D CNN, seed {s} =====')
    net = C.make_cnn1d(width=32)
    net, hist = C.train_deep(net, Xtr, Ytr, Xva, Y[va], epochs=40, lr=2e-3, bs=128, seed=s)
    pv, pt = C.predict(net, Xva), C.predict(net, Xte)
    cnn_val.append(pv); cnn_test.append(pt)
    row = C.summarize(f'cnn1d_s{s}', Y[va], pv, Y[te], pt, save_dir=PRED)
    results.append(row)
    torch.save(net.state_dict(), WORK + f'/cnn1d_s{s}.pt')
    print(f"seed {s}: test AUROC {row['AUROC']:.4f}  F1 {row['F1']:.4f}")

# average of the 3 seeds (a simple ensemble)
row = C.summarize('cnn1d_3seed_mean', Y[va], np.mean(cnn_val, 0), Y[te], np.mean(cnn_test, 0), save_dir=PRED)
results.append(row)
seed_aucs = [r['AUROC'] for r in results if r['model'].startswith('cnn1d_s')]
print(f'\n1D CNN across seeds: AUROC {np.mean(seed_aucs):.4f} +/- {np.std(seed_aucs):.4f}')
print(f"3-seed average:       AUROC {row['AUROC']:.4f}")

## 4. Results table

In [ ]:
res = pd.DataFrame(results).set_index('model')
res.to_csv(WORK + '/results_02_baselines.csv')
show = ['AUROC', 'AUROC_lo', 'AUROC_hi', 'AUPRC', 'F1']
print('Overall (test fold 10):')
display(res[show].round(4).sort_values('AUROC', ascending=False))
print('Per-class AUROC:')
display(res[[f'AUROC_{c}' for c in C.CLASSES]].round(4))
print('Per-class F1:')
display(res[[f'F1_{c}' for c in C.CLASSES]].round(4))

## 5. What to do with this
Send me the three tables. They are the baseline every later model has to beat. Next notebook: 1D ResNet (and xresnet / Inception variants).